# Competitive race (C — finals)

Before you start working with this notebook, remember to:

* Ensure the value of `max_time` is what you want.
* Put mp3 files in the `soundtrack` directory.

If you interrupt a cell and the simulator stops responding, you can try creating a new simulator (and new views).

Specify maximum length of each race (in seconds) before time-out.

In [ ]:
max_time = 120.

Import modules and configure the notebook.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import secrets
import json
import shutil
from pathlib import Path
from playsound3 import playsound
from matplotlib.patches import Rectangle
from matplotlib.backends.backend_agg import FigureCanvasAgg
import ae353_drone

Prevent students from importing `ae353_drone` in their own code.

In [ ]:
import sys
sys.modules['ae353_drone'] = None

Treat `RuntimeWarnings` as errors so that student code does not completely break the simulator (e.g., making it so collision checking stops working and drones fly through the ground).

In [ ]:
import warnings
warnings.filterwarnings('error', category=RuntimeWarning)

Enable playing music.

In [ ]:
class Soundtrack:
    def __init__(self, srcdir):
        self.playlist = []
        for file in os.listdir(srcdir):
            if file.endswith('.mp3'):
                self.playlist.append(os.path.join(srcdir, file))
        self.index = 0
        self.sound = None
    
    def start(self):
        if self.sound is not None:
            self.sound.stop()
        self.sound = playsound(self.playlist[self.index], block=False)
        self.index += 1
        if self.index == len(self.playlist):
            self.index = 0
    
    def stop(self):
        self.sound.stop()

soundtrack = Soundtrack('soundtrack')

Create and print seed so it is possible to reproduce the results.

In [ ]:
seed = secrets.randbits(32)
print(seed)

Create simulator.

In [ ]:
simulator = ae353_drone.Simulator(seed=seed)

Add camera views. (If you want to test this notebook quickly by running faster than real-time, replace the contents of this cell with a call to `simulator.disable_views()`.)

In [ ]:
simulator.add_view('my_start_view', 'start')
simulator.add_view('my_top_view', 'top')
simulator.add_view('my_side_view', 'right')
simulator.add_ring_view('my_ring_view', 7, yaw=15., distance=4.)

Load information about prelim race.

In [ ]:
race_information_path = Path('race-information.json')
with open(race_information_path, 'r') as infile:
    information = json.load(infile)
datetimestr = information['datetimestr']
teams = information['teams']
srcdir_designs = Path(f'{datetimestr}-designs')

Load student roster and list of finalists. Also clean up old files and folders, if they exist.

In [ ]:
# Get student roster (updated with prelim failures)
with open(Path(f'{datetimestr}-B-students.json'), 'r') as infile:
    students = json.load(infile)

# Get list of students who are finalists
with open(Path(f'{datetimestr}-B-finalists.json'), 'r') as infile:
    finalists = json.load(infile)
finalists = finalists['finalists']
if len(finalists) != 2 * len(teams):
    raise Exception('There must be two finalists per team.')

# Delete old files and folders, if they exist
for item in Path('.').iterdir():
    if item.name.startswith(f'{datetimestr}-C-race'):
        if item.is_file():
            item.unlink()
        elif item.is_dir():
            shutil.rmtree(item)
        else:
            raise Exception(f'{item.name} is neither a file nor a directory')

Create bracket.

In [ ]:
# Left bracket with one group per team, shuffled
bracket_L = np.array(finalists)[[0, 2, 4, 6]].tolist()
simulator.rng.shuffle(bracket_L)

# Right bracket with one group per team, shuffled
bracket_R = np.array(finalists)[[1, 3, 5, 7]].tolist()
simulator.rng.shuffle(bracket_R)

# Full bracket with empty lists for races whose
# competitors are still undetermined
bracket = {
    'quarterfinals': [
        [bracket_L[0], bracket_L[1]],
        [bracket_L[2], bracket_L[3]],
        [bracket_R[0], bracket_R[1]],
        [bracket_R[2], bracket_R[3]],
    ],
    'semifinals': [
        [],
        [],
    ],
    'finals': [
        [],
    ]
}

Define functions to get students by netid and partners by student.

In [ ]:
def get_student(students, netid):
    for student in students:
        if student['netid'] == netid:
            return student
    return None

def get_partners(students, student):
    partner_netids = np.array(student['dp4_partner']).flatten().tolist()
    partner_students = []
    for netid in partner_netids:
        partner_students.append(get_student(students, netid))
    return partner_students

Define functions to show results.

In [ ]:
benign_failures = [
    'Inactive.',
    'Out of bounds.',
]

def get_netids_to_email(drone_name, students):
    student = get_student(students, drone_name)
    if student is None:
        raise Exception(f'could not find student for this drone name: {drone_name}')
    
    netids_to_email = []
    netids_to_email.append(student['netid'] + '@illinois.edu')
    partners = get_partners(students, student)
    for partner in partners:
        netids_to_email.append(partner['netid'] + '@illinois.edu')
    
    return netids_to_email

def get_student_name(drone_name, students):
    student = get_student(students, drone_name)
    if student is None:
        raise Exception(f'could not find student for this drone name: {drone_name}')
    
    name = f'{student["first_name"]} {student["last_name"]}'
    partners = get_partners(students, student)
    for partner in partners:
        name += f' and {partner["first_name"]} {partner["last_name"]}'
        
    return name

def disqualify_student(drone_name, drone_error, students):
    student = get_student(students, drone_name)
    if student is None:
        raise Exception(f'could not find student for this drone name: {drone_name}')
    student['dp4_status'] = 'disqualified'
    student['dp4_error'] = drone_error
    partners = get_partners(students, student)
    for partner in partners:
        partner['dp4_status'] = 'disqualified'
        partner['dp4_error'] = drone_error

def get_results(simulator, students):
    netids_to_email = []
    finished = []
    still_running = []
    failed = []
    errors = ''
    results = ''
    for drone in simulator.drones:
        if drone['finish_time'] is not None:
            finished.append((drone, drone['finish_time']))
        elif drone['running']:
            still_running.append(drone)
        else:
            failed.append(drone)
            errors += f'======================\n{drone["error"]}\n======================\n\n'
    finished = sorted(finished, key=lambda f: f[1])
    
    results += 'FINISHED\n'
    for d in finished:
        drone = d[0]
        drone_name = drone['name']
        student_name = get_student_name(drone_name, students)
        results += f' {d[1]:6.2f} : {drone_name:20s} : {student_name}\n'

    results += '\nSTILL RUNNING\n'
    for d in still_running:
        drone = d
        drone_name = drone['name']
        student_name = get_student_name(drone_name, students)
        results += f'        : {drone_name:20s} : {student_name}\n'
    
    results += '\nINACTIVE OR OUT OF BOUNDS\n'
    for d in failed:
        drone = d
        drone_name = drone['name']
        if drone['error'] in benign_failures:
            student_name = get_student_name(drone_name, students)
            results += f'        : {drone_name:20s} : {student_name}\n'
    
    results += '\nFAILED\n'
    for d in failed:
        drone = d
        drone_name = drone['name']
        if drone['error'] not in benign_failures:
            disqualify_student(drone_name, drone['error'], students)
            student_name = get_student_name(drone_name, students)
            netids_to_email.extend(get_netids_to_email(drone_name, students))
            results += f'        : {drone_name:20s} : {student_name}\n'
    
    results += '\nERRORS (REASONS FOR FAILURE)\n\n'
    results += errors
    
    results += '\nNETIDS TO EMAIL ABOUT FAILURE\n\n'
    results += (' ' + ', '.join(netids_to_email))
    
    return results

Define functions to prepare and run final races.

In [ ]:
def get_team(racer):
    student = get_student(students, racer)
    assert(student is not None)
    return student['dp4_team']

def get_racer_one(racers):
    racer_one = racers[0]
    racers.remove(racer_one)
    return racer_one, get_team(racer_one)

def get_racer_two(racers, team):
    for racer_two in racers:
        if team == get_team(racer_two):
            racers.remove(racer_two)
            return racer_two
    return None

def count_on_team(racers, team):
    count = 0
    for racer in racers:
        if get_team(racer) == team:
            count += 1
    return count

def get_racers_on_team(racers, team):
    racers_on_team = []
    for racer in racers:
        if get_team(racer) == team:
            racers_on_team.append(racer)
    return racers_on_team

def get_racer_label(students, racer):
    student = get_student(students, racer)
    name = f'{student["first_name"]} {student["last_name"]}'
    partners = get_partners(students, student)
    for partner in partners:
        name += f'\n{partner["first_name"]} {partner["last_name"]}'
    team = student['dp4_team']
    name += f'\n\nTEAM {team.upper()}'
    return name, teams[team]

def get_racer_status(drone):
    if drone['finish_time'] is not None:
        return f'FINISHED ({drone['finish_time']:.2f})'
    if drone['running']:
        return 'TIMED OUT'
    return 'CRASHED'

def get_winning_racer(drones):
    winning_name = None
    winning_time = np.inf
    for drone in drones:
        if drone['finish_time'] is None:
            continue
        if drone['finish_time'] < winning_time:
            winning_name = drone['name']
            winning_time = drone['finish_time']
    return winning_name

def show_title_frame(drones, students, finish=False, width=640, height=480, dpi=100):
    if len(drones) != 2:
        raise Exception('There must be exactly two drones in the race.')

    drone_one = drones[0]
    drone_two = drones[1]
    racer_one = drone_one['name']
    racer_two = drone_two['name']
    path_png_one = Path(drone_one['image'])
    path_png_two = Path(drone_two['image'])

    width = 640
    height = 480
    dpi = 100

    dx = 40
    y = 200
    w = 200

    fig = plt.figure(figsize=(width/dpi, height/dpi), dpi=dpi)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlim([0, width])
    ax.set_ylim([0, height])
    ax.set_aspect('equal')
    im = plt.imread(path_png_one)
    ax.imshow(im, aspect='equal', extent=(dx, dx + w, y, y + w))
    label_one, rgb_one = get_racer_label(students, racer_one)
    ax.text(dx + (w / 2), y - dx, label_one, ha='center', va='top', fontsize=12)
    ax.add_patch(Rectangle((dx + (w / 2) - (dx / 2), dx / 2), dx, dx, linewidth=0, facecolor=rgb_one))
    im = plt.imread(path_png_two)
    ax.imshow(im, aspect='equal', extent=(width - (w + dx), width - dx, y, y + w))
    label_two, rgb_two = get_racer_label(students, racer_two)
    ax.text(width - (dx + (w / 2)), y - dx, label_two, ha='center', va='top', fontsize=12)
    ax.add_patch(Rectangle((width - (dx + (w / 2)) - (dx / 2), dx / 2), dx, dx, linewidth=0, facecolor=rgb_two))
    ax.text(width / 2, y + w / 2, 'versus', ha='center', va='center', fontsize=18)

    if finish:
        ax.text(
            dx + (w / 2), y + w + (dx / 4),
            get_racer_status(drone_one),
            ha='center', va='bottom', fontsize=12,
        )
        ax.text(
            width - (dx + (w / 2)), y + w + (dx / 4),
            get_racer_status(drone_two),
            ha='center', va='bottom', fontsize=12,
        )
        winning_racer = get_winning_racer(drones)
        if winning_racer == racer_one:
            ax.text(
                dx + (w / 2), height - dx,
                'WINNER',
                ha='center', va='bottom', fontsize=16, weight='bold',
            )
        elif winning_racer == racer_two:
            ax.text(
                width - (dx + (w / 2)), height - dx,
                'WINNER',
                ha='center', va='bottom', fontsize=16, weight='bold',
            )
        else:
            ax.text(
                width / 2, height - dx,
                'NO WINNER (VOID RACE)',
                ha='center', va='bottom', fontsize=16, weight='bold',
            )
    
    plt.show()

def finals_need_race(racer_one, racer_two):
    if (racer_one is None) and (racer_two is None):
        print('VOID RACE (NO COMPETITORS)')
        return False, None
    if (racer_two is None):
        print('WINNER BY DEFAULT:\n')
        label, color = get_racer_label(students, racer_one)
        print(label)
        return False, racer_one
    if (racer_one is None):
        print('WINNER BY DEFAULT:\n')
        label, color = get_racer_label(students, racer_two)
        print(label)
        return False, racer_two
    else:
        return True, None

def finals_prepare_race(racer_one, racer_two, index_of_race):
    # Get files ready
    dstdir = f'{datetimestr}-C-race-{index_of_race:03d}-{racer_one}-{racer_two}'
    os.mkdir(dstdir)
    for racer in [racer_one, racer_two]:
        shutil.copyfile(
            os.path.join(srcdir_designs, f'{racer}.py'),
            os.path.join(dstdir, f'{racer}.py'),
        )
        shutil.copyfile(
            os.path.join(srcdir_designs, f'{racer}.png'),
            os.path.join(dstdir, f'{racer}.png'),
        )

    # Get simulator ready
    simulator.clear_drones()
    simulator.place_rings()
    simulator.load_drones(dstdir)
    simulator.reset()

    # Show title frame
    show_title_frame(simulator.drones, students, finish=False)

    return dstdir

def finals_run_race(dstdir):
    # Run simulator
    soundtrack.start()
    simulator.run(max_time=max_time, print_debug=True)
    soundtrack.stop()

    # Find winner
    winning_name = None
    winning_time = np.inf
    for drone in simulator.drones:
        if drone['finish_time'] is None:
            continue
        if drone['finish_time'] < winning_time:
            winning_name = drone['name']
            winning_time = drone['finish_time']
    
    # Get and write results (also, update students to reflect disqualifications)
    results = get_results(simulator, students)
    print('\n\n================RESULTS================\n\n')
    print(results)
    with open(f'{dstdir}.txt', 'w') as f:
        f.write(results)

    # Show title frame
    show_title_frame(simulator.drones, students, finish=True)

    return winning_name

Initialize race index.

In [ ]:
index_of_race = 0

## Quarterfinals

### Quarterfinal 1/4

Prepare race.

In [ ]:
index_of_race += 1
racer_one, racer_two = bracket['quarterfinals'][0]
dstdir = finals_prepare_race(racer_one, racer_two, index_of_race)

Run race.

In [ ]:
winner = finals_run_race(dstdir)

Record results (you may want to repeat "prepare race" and "run race" before proceeding, if you get a void race).

In [ ]:
bracket['semifinals'][0].append(winner)

### Quarterfinal 2/4

Prepare race.

In [ ]:
index_of_race += 1
racer_one, racer_two = bracket['quarterfinals'][1]
dstdir = finals_prepare_race(racer_one, racer_two, index_of_race)

Run race.

In [ ]:
winner = finals_run_race(dstdir)

Record results.

In [ ]:
bracket['semifinals'][0].append(winner)

### Quarterfinal 3/4

Prepare race.

In [ ]:
index_of_race += 1
racer_one, racer_two = bracket['quarterfinals'][2]
dstdir = finals_prepare_race(racer_one, racer_two, index_of_race)

Run race.

In [ ]:
winner = finals_run_race(dstdir)

Record results.

In [ ]:
bracket['semifinals'][1].append(winner)

### Quarterfinal 4/4

Prepare race.

In [ ]:
index_of_race += 1
racer_one, racer_two = bracket['quarterfinals'][3]
dstdir = finals_prepare_race(racer_one, racer_two, index_of_race)

Run race.

In [ ]:
winner = finals_run_race(dstdir)

Record results.

In [ ]:
bracket['semifinals'][1].append(winner)

## Semifinals

### Semifinal 1/2

Decide if a race is needed.

In [ ]:
racer_one, racer_two = bracket['semifinals'][0]
need_race, winner = finals_need_race(racer_one, racer_two)

Prepare race.

In [ ]:
if need_race:
    index_of_race += 1
    dstdir = finals_prepare_race(racer_one, racer_two, index_of_race)

Run race.

In [ ]:
if need_race:
    winner = finals_run_race(dstdir)

Record results.

In [ ]:
bracket['finals'][0].append(winner)

### Semifinal 2/2

Decide if a race is needed.

In [ ]:
racer_one, racer_two = bracket['semifinals'][1]
need_race, winner = finals_need_race(racer_one, racer_two)

Prepare race.

In [ ]:
if need_race:
    index_of_race += 1
    dstdir = finals_prepare_race(racer_one, racer_two, index_of_race)

Run race.

In [ ]:
if need_race:
    winner = finals_run_race(dstdir)

Record results.

In [ ]:
bracket['finals'][0].append(winner)

## Finals

Decide if a race is needed.

In [ ]:
racer_one, racer_two = bracket['finals'][0]
need_race, winner = finals_need_race(racer_one, racer_two)

Prepare race.

In [ ]:
if need_race:
    index_of_race += 1
    dstdir = finals_prepare_race(racer_one, racer_two, index_of_race)

Run race.

In [ ]:
if need_race:
    winner = finals_run_race(dstdir)

## Save results

Save results of finals to file.

In [ ]:
# Student roster (updated with finals failures)
with open(f'{datetimestr}-C-students.json', 'w') as outfile:
    json.dump(students, outfile, indent=4)

# Information about race (updated with finals information)
information['seed-C'] = seed
with open(race_information_path, 'w') as outfile:
    json.dump(information, outfile, indent=4,)